In [1]:
# SE IMPORTAN LIBRERIAS DE TRABAJO
#-----------------------------------

import pandas as pd
import requests
import os
import io
import unicodedata #para manejo de tildes


In [2]:
# NAVEGA POR RELEASES de GITHUB PARA ENCONTRAR LOS ARCHIVOS EXISTENTES

user = "Marcelo-F-Martin"
repo = "PI_UA_pipeline_analisis_de_cobranza"

# URL de la API para la última release
api_url = f"https://api.github.com/repos/{user}/{repo}/releases/latest"

response = requests.get(api_url)
assets = response.json().get('assets', []) # [] es el segundo argumento del .get(). Significa: "Si por alguna razón la llave 'assets' no existe, no rompas el programa, devuélveme una lista vacía".
assets


[{'url': 'https://api.github.com/repos/Marcelo-F-Martin/PI_UA_pipeline_analisis_de_cobranza/releases/assets/375078221',
  'id': 375078221,
  'node_id': 'RA_kwDORolAKc4WWz1N',
  'name': 'Detalle.de.comisiones.Broker.xls',
  'label': None,
  'uploader': {'login': 'Marcelo-F-Martin',
   'id': 267367303,
   'node_id': 'U_kgDOD--zhw',
   'avatar_url': 'https://avatars.githubusercontent.com/u/267367303?v=4',
   'gravatar_id': '',
   'url': 'https://api.github.com/users/Marcelo-F-Martin',
   'html_url': 'https://github.com/Marcelo-F-Martin',
   'followers_url': 'https://api.github.com/users/Marcelo-F-Martin/followers',
   'following_url': 'https://api.github.com/users/Marcelo-F-Martin/following{/other_user}',
   'gists_url': 'https://api.github.com/users/Marcelo-F-Martin/gists{/gist_id}',
   'starred_url': 'https://api.github.com/users/Marcelo-F-Martin/starred{/owner}{/repo}',
   'subscriptions_url': 'https://api.github.com/users/Marcelo-F-Martin/subscriptions',
   'organizations_url': 'https

## Inicia proceso ETL
#### 1- Extracción:
#####   Obtención de datos crudos desde la fuente y almacenamiento en Dataframe de Pandas

In [3]:
df_definitivo_por_libro = []   
df_final = []

#====== INICIO 2º bucle: recorre hojas del archivo ===============================

for asset in assets:
    
    # Filtramos para que solo lea los archivos .xls
    if asset['name'].endswith('.xls'):
        print(f"Detectado archivo: {asset['name']}")
        
        # El campo 'browser_download_url' es el link directo
        
    hojas_consolidadas_por_libro = [] # se crea una lista para incorporar los df por cada hoja de cada libro, para luego agrupar todos.
    
    file_url = asset['browser_download_url']
    file_data = requests.get(file_url).content
    
    ## df_temp = pd.read_excel(io.BytesIO(file_data), engine='xlrd') -----se incorpora esta linea en la linea siguiente.

    hojas_brutas = pd.read_excel(io.BytesIO(file_data), engine='xlrd', sheet_name=None, header=None)

#====== INICIO 1º bucle: recorre hojas del archivo ===============================
        
    for nombre_hoja, df_bruto in hojas_brutas.items():
                   
        # Aquí 'df_bruto' es el DataFrame bruto de la hoja actual
        print(f"✅ Hoja '{nombre_hoja}' cargada correctamente. Filas brutas: {len(df_bruto)}")
        # Retornamos el diccionario para el siguiente paso del procesamiento

        print(f"  --- Buscando clave en Hoja '{nombre_hoja}' ---")
        
        encabezado_clave = "Periodo"
        indice_encabezado_fila = df_bruto[df_bruto.apply(lambda row: encabezado_clave in row.astype(str).values, axis=1)].index
        
        fila_inicio = indice_encabezado_fila[0]
        print(f"✅ Encabezado clave '{encabezado_clave}' encontrado en la Fila (índice 0): {fila_inicio}")

        fila_encabezado = df_bruto.iloc[fila_inicio]

        # 2b. 'col_inicio' será el índice de la columna donde se encuentra la clave
        # Esto es crucial para ignorar las columnas añadidas a la izquierda.
        
        col_inicio = fila_encabezado[fila_encabezado.astype(str) == encabezado_clave].index[0]
        print(f"  ✅ Columna de inicio de datos (índice 0): {col_inicio}")

        # Por eficiencia, si ya tienes el df_bruto, lo mejor es usarlo para el recorte:
        df_bruto.columns = df_bruto.iloc[fila_inicio] # Asignar la fila encontrada como nuevo encabezado
        df_limpio = df_bruto[fila_inicio + 1:].reset_index(drop=True) # Eliminar filas superiores

        df_final_hoja = df_limpio.copy()
        df_final_hoja['hoja_origen'] = nombre_hoja # se inserta nueva columna que identifica la Hoja de Origen

        if not df_final_hoja.empty:
            hojas_consolidadas_por_libro.append(df_final_hoja)
            print(f"  ✅ Hoja '{nombre_hoja}' añadida a la consolidación. Registros: {len(df_final_hoja)}")
        else:
            print(f"  Advertencia: Hoja '{nombre_hoja}' vacía después de la limpieza. No se añadió.")


       
    df_archivo_unificado = pd.concat(hojas_consolidadas_por_libro, ignore_index=True)
    df_definitivo_por_libro.append(df_archivo_unificado)
            
#====== FIN 1º bucle: recorre hojas del archivo ===============================
#====== FIN 2º bucle: recorre hojas del archivo ===============================

# Unificación final
df_final = pd.concat(df_definitivo_por_libro, ignore_index=True)


Detectado archivo: Detalle.de.comisiones.Broker.xls
✅ Hoja 'AI004' cargada correctamente. Filas brutas: 33008
  --- Buscando clave en Hoja 'AI004' ---
✅ Encabezado clave 'Periodo' encontrado en la Fila (índice 0): 7
  ✅ Columna de inicio de datos (índice 0): 0
  ✅ Hoja 'AI004' añadida a la consolidación. Registros: 33000
Detectado archivo: Detalle.de.comisiones.Carlos.Diaz.xls
✅ Hoja 'AI002' cargada correctamente. Filas brutas: 787
  --- Buscando clave en Hoja 'AI002' ---
✅ Encabezado clave 'Periodo' encontrado en la Fila (índice 0): 7
  ✅ Columna de inicio de datos (índice 0): 0
  ✅ Hoja 'AI002' añadida a la consolidación. Registros: 779
✅ Hoja 'AI003' cargada correctamente. Filas brutas: 754
  --- Buscando clave en Hoja 'AI003' ---
✅ Encabezado clave 'Periodo' encontrado en la Fila (índice 0): 7
  ✅ Columna de inicio de datos (índice 0): 0
  ✅ Hoja 'AI003' añadida a la consolidación. Registros: 746
✅ Hoja 'AI013' cargada correctamente. Filas brutas: 640
  --- Buscando clave en Hoja '

In [4]:
df_final.head()

7,Periodo,Region,Asesor,Denominación Asesor,Broker,ID Broker,Fecha Operacion,Contrato,Rama,Especialidad,ID Cliente,Nombre Cliente,Comision,Valor Neto,Porcentaje Comision,Forma Pago,Numero Recibo,hoja_origen
0,202506,A,AE047,Sebastián Franco,B0001,Broker Health Co.,2025-06-11 00:00:00,GUPW9,Oftalmología,ESP22,CL01790L,Martin Acosta,1730.9665,34619.33,0.05,efe,24807,AI004
1,202510,A,AE006,Patricia Vega,B0001,Broker Health Co.,2025-10-20 00:00:00,JN4TM,Oftalmología,ESP22,CL01759E,Patricia Aguirre,1906.9085,38138.17,0.05,transf,78856,AI004
2,202511,A,AE006,Patricia Vega,B0001,Broker Health Co.,2025-11-23 00:00:00,WKB45,Oftalmología,ESP22,CL00210T,Grupo Aguirre SA,1558.109,31162.18,0.05,transf,71496,AI004
3,202503,A,AE011,Héctor Farías,B0001,Broker Health Co.,2025-03-29 00:00:00,55T20,Pediatría,ESP08,CL00655F,Laboratorio Quiroga SA,1673.154,16731.54,0.1,efe,81111,AI004
4,202508,A,AE026,Camila Paredes,B0001,Broker Health Co.,2025-08-09 00:00:00,VK0Q0,Oftalmología,ESP24,CL00439E,Martin Sosa,1598.249,15982.49,0.1,transf,38615,AI004


#### 2- Exploración y Transformación

In [6]:
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 18 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Periodo              46000 non-null  object
 1   Region               46000 non-null  object
 2   Asesor               46000 non-null  object
 3   Denominación Asesor  46000 non-null  object
 4   Broker               46000 non-null  object
 5   ID Broker            46000 non-null  object
 6   Fecha Operacion      46000 non-null  object
 7   Contrato             46000 non-null  object
 8   Rama                 46000 non-null  object
 9   Especialidad         46000 non-null  object
 10  ID Cliente           46000 non-null  object
 11  Nombre Cliente       46000 non-null  object
 12  Comision             46000 non-null  object
 13  Valor Neto           46000 non-null  object
 14  Porcentaje Comision  46000 non-null  object
 15  Forma Pago           46000 non-null  object
 16  Nume

In [7]:
def limpiar_tilde(texto):
    # 1. Normaliza el texto a la forma NFD (Descomposición)
    # Esto separa la 'á' en 'a' + 'tilde combinable'
    texto_normalizado = unicodedata.normalize('NFD', texto)
    
    # 2. Filtra y mantiene solo los caracteres que no sean "marcas" de acento (Mn)
    texto_sin_acentos = "".join(
        c for c in texto_normalizado if unicodedata.category(c) != 'Mn'
    )
    
    return texto_sin_acentos

In [8]:
'''
1_estandariza a minusculas
2_elimina espacios en blanco
3_reemplaza vacíos por '_'
4_aplica función definida para limpiar tildes
________________________________________________
'''

df_final.columns = df_final.columns.str.lower().str.strip().str.replace(' ','_')
df_final.columns = [limpiar_tilde(col) for col in df_final.columns]
df_final.columns

Index(['periodo', 'region', 'asesor', 'denominacion_asesor', 'broker',
       'id_broker', 'fecha_operacion', 'contrato', 'rama', 'especialidad',
       'id_cliente', 'nombre_cliente', 'comision', 'valor_neto',
       'porcentaje_comision', 'forma_pago', 'numero_recibo', 'hoja_origen'],
      dtype='object')

In [9]:
# seleccion de columnas relevantes a importar a la Base de Datos
lista_columnas_seleccionadas = ['asesor', 'broker', 'fecha_operacion','contrato','comision', 'valor_neto', 'porcentaje_comision','forma_pago', 'numero_recibo'  ]
df_final = df_final[lista_columnas_seleccionadas]

In [10]:
df_final

,asesor,broker,fecha_operacion,contrato,comision,valor_neto,porcentaje_comision,forma_pago,numero_recibo
0,AE047,B0001,2025-06-11 00:00:00,GUPW9,1730.9665,34619.33,0.05,efe,24807
1,AE006,B0001,2025-10-20 00:00:00,JN4TM,1906.9085,38138.17,0.05,transf,78856
2,AE006,B0001,2025-11-23 00:00:00,WKB45,1558.109,31162.18,0.05,transf,71496
3,AE011,B0001,2025-03-29 00:00:00,55T20,1673.154,16731.54,0.1,efe,81111
4,AE026,B0001,2025-08-09 00:00:00,VK0Q0,1598.249,15982.49,0.1,transf,38615
...,...,...,...,...,...,...,...,...,...
45995,AI004,B0001,2025-10-26 00:00:00,RVUES,3357.106,16785.53,0.2,tc,18246
45996,AI004,B0001,2025-01-03 00:00:00,3FKC9,706.393,7063.93,0.1,transf,63092
45997,AI004,B0001,2025-03-02 00:00:00,H75FR,8387.84,41939.2,0.2,transf,86040
45998,AI004,B0001,2025-08-20 00:00:00,RYSTM,3505.787,35057.87,0.1,efe,77539


In [11]:
# modifica tipo de datos

df_final = df_final.astype({'comision': float, 'valor_neto': float, 'porcentaje_comision': float})
df_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 46000 entries, 0 to 45999
Data columns (total 9 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   asesor               46000 non-null  object 
 1   broker               46000 non-null  object 
 2   fecha_operacion      46000 non-null  object 
 3   contrato             46000 non-null  object 
 4   comision             46000 non-null  float64
 5   valor_neto           46000 non-null  float64
 6   porcentaje_comision  46000 non-null  float64
 7   forma_pago           46000 non-null  object 
 8   numero_recibo        46000 non-null  object 
dtypes: float64(3), object(6)
memory usage: 3.2+ MB


In [12]:
round(df_final.describe(),2)

,comision,valor_neto,porcentaje_comision
count,46000.00,46000.00,46000.00
mean,3298.17,25251.81,0.13
std,2713.10,14267.87,0.07
min,25.95,500.25,0.05
25%,1298.19,12926.42,0.10
50%,2570.10,25168.93,0.10
75%,4526.65,37736.74,0.15
max,14990.63,49998.89,0.30


In [13]:
# Verifica existencia de PK
df_final.nunique()

asesor                    54
broker                     1
fecha_operacion          363
contrato                3000
comision               45899
valor_neto             32889
porcentaje_comision        5
forma_pago                 3
numero_recibo          15014
dtype: int64

In [14]:
# Esta celda puede ser opcional para exportar el dataframe en CSV.
ruta_archivo = r"C:\Users\orglo\OneDrive\Desktop\marcelo\Proyectos\PI"
nombre_csv_salida = f"comisiones_consolidadas.csv"
ruta_salida_completa = os.path.join(ruta_archivo, nombre_csv_salida)
df_final.to_csv(ruta_salida_completa, index=False, encoding='utf-8')